In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [20]:
data=pd.read_excel('../data/data.xlsx',sheet_name="Online Retail")

In [ ]:
print("SHape :",data.shape)
print("\nHead :",data.head())

In [ ]:
print("Data types:",data.dtypes)
print("\nDescription :")
print(data.describe(include="all"))

In [ ]:
missing=data.isnull().sum()

duplicates=data.duplicated().sum()

print("Missing values:\n",missing)

print("\nNumber of exact duplicates:", duplicates)

In [ ]:
cancelled = data["InvoiceNo"].astype(str).str.startswith("C")
print("Cancelled invoices:", cancelled.sum())

In [ ]:
invalid_quantity = data["Quantity"] <= 0

print("Quantity <= 0:", invalid_quantity.sum())

In [ ]:
invalid_price = data["UnitPrice"] <= 0

print("UnitPrice <= 0:", invalid_price.sum())

In [ ]:
anomalies=pd.DataFrame({
    "Anomaly":[
        "Cancelled Invoices",
        "Quantity<=0",
        "UnitPrice<=0"
    ],
    "Number of rows":[
        cancelled.sum(),
        invalid_price.sum(),
        invalid_quantity.sum()
    ]
})

print(anomalies)

In [ ]:
data_clean=data.copy()

data_clean = data_clean.dropna(subset=["CustomerID"])

data_clean=data_clean.drop_duplicates()

data_clean = data_clean[
    ~data_clean["InvoiceNo"].astype(str).str.startswith("C")
]

data_clean = data_clean[data_clean["Quantity"] > 0]

data_clean = data_clean[data_clean["UnitPrice"] > 0]

data_clean["InvoiceDate"] = pd.to_datetime(data_clean["InvoiceDate"])

data_clean["TotalPrice"] = (
    data_clean["Quantity"] * data_clean["UnitPrice"]
)

print(data_clean.shape)
print(data_clean.head())

In [ ]:
#Transactions by country
country_count=data_clean["Country"].value_counts().head(10)
plt.figure(figsize=(10,5))
sns.barplot(x=country_count.values, y=country_count.index)
plt.title("Top 10 des pays par nombre de transactions")
plt.xlabel("Nombre de transactions")
plt.ylabel("Pays")
plt.show()

In [ ]:
#Transactions by month

monthly=(data_clean.set_index("InvoiceDate").resample("ME").size())
plt.figure(figsize=(10, 5))
monthly.plot()
plt.title("Nombre de transactions par mois")
plt.xlabel("Mois")
plt.ylabel("Nombre de transactions")
plt.show()

In [ ]:
#Top products
top_products=(data_clean.groupby("Description")["Quantity"].sum().sort_values(ascending=False).head(10))
plt.figure(figsize=(10, 5))
sns.barplot(x=top_products.values,y=top_products.index)
plt.title("Top 10 des produits les plus vendus")
plt.xlabel("Quantité vendue")
plt.ylabel("Produit")
plt.show()